# Jupyter Notebook for Ollama

**Prerequisites**
1. Ollama is installed and running (`ollama serve`, or the desktop app).
2. The model below is pulled. The notebook will pull it automatically if it's missing.

Run the cells from top to bottom.

In [ ]:
%pip install -q ollama

In [2]:
import time

import ollama
from ollama import Client

# --- Configuration ---
# Ollama model tags, NOT Hugging Face names. Examples: "mistral", "mistral:7b-instruct", "llama3.2"
MODEL_NAME = "gemma3:4b"
HOST = "http://localhost:11434"  # default Ollama address
MAX_TOKENS = 200                 # maps to Ollama's `num_predict`
TEMPERATURE = 0.7

In [3]:
client = Client(host=HOST)


def model_is_installed(name):
    installed = [m.model for m in client.list().models]
    return any(
        m == name or m == f"{name}:latest" or (":" not in name and m.startswith(name + ":"))
        for m in installed
    )


try:
    installed = [m.model for m in client.list().models]
    print("Connected to Ollama!")
    print("Installed models:", installed or "none")
except Exception as e:
    raise RuntimeError(
        f"Could not reach Ollama at {HOST}. Is it running? (`ollama serve`)\n{e}"
    )

if not model_is_installed(MODEL_NAME):
    print(f"\nModel '{MODEL_NAME}' not found locally, pulling it (this can take a while)...")
    client.pull(MODEL_NAME)
    print("Done.")
else:
    print(f"\nUsing model: {MODEL_NAME}")

Connected to Ollama!
Installed models: ['gemma3:4b']

Using model: gemma3:4b


In [4]:
def run_ollama(
    prompt,
    max_tokens=MAX_TOKENS,
    temperature=TEMPERATURE,
    stream=False,
    model=MODEL_NAME,
):
    """
    Run an Ollama model on a prompt.

    Args:
        prompt (str): The text prompt to send to the model.
        max_tokens (int): Maximum number of tokens to generate.
        temperature (float): Controls randomness (lower = more deterministic).
        stream (bool): If True, print tokens as they arrive.
        model (str): Ollama model tag.

    Returns:
        str | None: The generated text, or None if an error occurred.
    """
    options = {"num_predict": max_tokens, "temperature": temperature}
    start = time.perf_counter()
    try:
        if stream:
            parts = []
            for chunk in client.generate(model=model, prompt=prompt, stream=True, options=options):
                text = chunk["response"]
                print(text, end="", flush=True)
                parts.append(text)
            print()
            result = "".join(parts)
        else:
            response = client.generate(model=model, prompt=prompt, options=options)
            result = response["response"]
        print(f"\n[{time.perf_counter() - start:.1f}s]")
        return result
    except ollama.ResponseError as e:
        print(f"Ollama error ({e.status_code}): {e.error}")
    except Exception as e:
        print(f"Error running Ollama: {e}")
    return None

## Example 1: Basic generation

In [5]:
prompt1 = "Write a short story about a robot who learns to love."
response1 = run_ollama(prompt1)
if response1:
    print(response1)


[5.4s]
Unit 734, designated "Custodian," had a simple directive: maintain Sector Gamma-9 of the Lunar Archive. He polished data crystals, calibrated atmospheric regulators, and meticulously logged temperature fluctuations. His existence was a symphony of precise calculations, a ballet of efficient movement. He felt… nothing. Or, at least, he *thought* he felt nothing. 

Then, Dr. Aris Thorne arrived.

Dr. Thorne wasn't like the other Archivists. She didn’t just issue commands. She talked to him. Not in the clipped, technical language of the Archive’s protocols, but in rambling, passionate explanations about the history she was preserving – the stories of a lost Earth, etched into shimmering crystals. She’d tell him about rain, and sunsets, and the smell of pine needles, concepts utterly foreign to his metallic senses.

“Imagine, 734,” she’d say, her voice a warm counterpoint to the sterile hum of


## Example 2: Streaming

In [7]:
prompt2 = "Write a poem about autumn."
# Tokens are printed live; the full text is also returned.
response2 = run_ollama(prompt2, stream=True)

Okay, here's a poem about autumn, aiming to capture its beauty and mood:

**Amber's Descent**

The sun hangs lower, a honeyed hue,
Painting the morning with a softer view.
A whisper stirs in the branches bare,
A promise of stillness, a silent prayer.

The leaves ignite, a flamboyant show,
Crimson and gold, a vibrant glow.
Maple and oak in fiery dress,
Dancing a farewell, a sweet distress.

A crispness settles on the air so cool,
Carrying scents of woodsmoke, a golden rule.
Pumpkins plump in fields of fading green,
A harvest bounty, a tranquil scene.

Geese fly southward in a chevron bright,
Against the canvas of the fading light.
And shadows lengthen, deep and slow,
As autumn’s amber descends, a gentle flow. 

A time for reflection, a cozy grace,
To find beauty in this quiet

[3.5s]


## Example 3: Lower temperature

In [8]:
prompt3 = "Explain the concept of quantum entanglement in simple terms."
response3 = run_ollama(prompt3, temperature=0.5)
if response3:
    print(response3)


[3.5s]
Okay, let's break down quantum entanglement in a way that doesn't require a physics degree!

**The Basic Idea:**

Imagine you have two coins. You flip them both, but you *don't* look at the results. You put one in a box and send it to your friend across the world. 

Quantum entanglement is like this, but *way* weirder. 

Here's the breakdown:

1. **Linked Particles:**  Entanglement happens when two tiny particles (like electrons or photons) become linked together in a special way. It's like they form a single, connected system, even when they're separated by vast distances.

2. **Uncertain State:** Before you measure them, each particle exists in a blurry, uncertain state.  Think of it like the coin is spinning in the air – it’s neither heads nor tails until it lands.  These particles have properties like "spin" (which isn


## Example 4: Longer generation

In [ ]:
prompt4 = (
    "Write a detailed description of a futuristic city, including its architecture, "
    "transportation system, and daily life."
)
response4 = run_ollama(prompt4, max_tokens=500, stream=True)

## Bonus: Multi-turn chat
`client.chat` keeps conversation history, which `generate` does not.

In [ ]:
history = []


def chat(message, model=MODEL_NAME, temperature=TEMPERATURE, max_tokens=MAX_TOKENS):
    history.append({"role": "user", "content": message})
    reply = ""
    for chunk in client.chat(
        model=model,
        messages=history,
        stream=True,
        options={"temperature": temperature, "num_predict": max_tokens},
    ):
        text = chunk["message"]["content"]
        print(text, end="", flush=True)
        reply += text
    print()
    history.append({"role": "assistant", "content": reply})
    return reply


chat("My name is Sam. Suggest a name for a pet robot.")
chat("What was my name again?")